# **MODEL 1: Multimodal Integration - Late Fusion (EfficientNetB1 + Metadata MLP)**

**Objective:**
Inclusion of **clinical metadata**
- **Age**
- **Sex**
- **Atomic location** of the lesion
<br>

**Methodology:**
| Variable | Strategy | NA Treatment |
|----------|-----------|--------------------|
| Age | Normalization (min-max) | Imputation by median |
| Sex | One-hot encoding | Category "unknown" |
| Location | One-hot encoding (15 sítios) | Category "unknown" |

### Imports

In [3]:
import os, sys, json
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

### **Data** Configuration

In [4]:
BASE_PATH = "./data"
AUG_IMG_DIR = os.path.join(BASE_PATH, "HAM10000_augmented")
AUG_META_PATH = os.path.join(BASE_PATH, "augmented_metadata.csv")
VAL_META_PATH = os.path.join(BASE_PATH, "val_split.csv")
TEST_META_PATH = os.path.join(BASE_PATH, "test_split.csv")

In [5]:
train_df = pd.read_csv(AUG_META_PATH)
val_df = pd.read_csv(VAL_META_PATH)
test_df = pd.read_csv(TEST_META_PATH)

FileNotFoundError: [Errno 2] No such file or directory: './data\\augmented_metadata.csv'

### **Data** Loading & Mapping

In [ ]:
# 2. UNIFIED METADATA PREPROCESSING
# Errors often occur here if validation encoding differs from training
def preprocess_metadata(df, scaler=None, encoder=None, is_training=True):
    # Handle missing values as defined in your methodology
    df['age'] = df['age'].fillna(df['age'].median())
    df['sex'] = df['sex'].fillna('unknown')
    df['localization'] = df['localization'].fillna('unknown')
    
    # Normalize Age
    if is_training:
        scaler = MinMaxScaler()
        df['age_scaled'] = scaler.fit_transform(df[['age']])
    else:
        df['age_scaled'] = scaler.transform(df[['age']])
        
    # One-Hot Encode Sex and Localization (15 sites)
    cat_cols = ['sex', 'localization']
    if is_training:
        encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')
        encoded_cats = encoder.fit_transform(df[cat_cols])
    else:
        encoded_cats = encoder.transform(df[cat_cols])
    
    # Combine into a single meta-vector
    meta_vectors = np.hstack([df[['age_scaled']].values, encoded_cats])
    return meta_vectors, scaler, encoder

## Metadata preprocessing

Clean and encode the clinical features (Age, Sex, Localization) into a fixed-length vector before the split.

In [ ]:
train_meta, scaler, encoder = preprocess_metadata(train_df, is_training=True)
val_meta, _, _ = preprocess_metadata(val_df, scaler, encoder, is_training=False)
test_meta, _, _ = preprocess_metadata(test_df, scaler, encoder, is_training=False)

MLP input dim: 1  →  ['age_norm']


## 4. Multimodal Data Pipeline

In [ ]:
# This ensures Stream A (Image) and Stream B (Meta) stay connected
def load_multimodal_item(path, meta, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [240, 240]) # EfficientNetB1 standard
    img = tf.cast(img, tf.float32) / 255.0
    return {"image_input": img, "meta_input": meta}, label

In [ ]:
def create_dataset(df, meta, batch_size=32):
    dataset = tf.data.Dataset.from_tensor_slices((df['image_path'].values, meta, df['dx_encoded'].values))
    dataset = dataset.map(load_multimodal_item).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

In [ ]:
train_ds = create_dataset(train_df, train_meta)
val_ds = create_dataset(val_df, val_meta)
test_ds = create_dataset(test_df, test_meta)

## 5. Architecture Late Fusion: EfficientNetB1 + Metadata MLP 

This builds the two branches and concatenates them into the final classification head. Establishing correct connections between EfficientNet and the MLP

In [ ]:
# Image Stream (EfficientNetB1)
base_model = tf.keras.applications.EfficientNetB1(include_top=False, weights='imagenet', input_shape=(240, 240, 3))
image_input = layers.Input(shape=(240, 240, 3), name="image_input")
x = base_model(image_input)
x = layers.GlobalAveragePooling2D()(x)
image_features = layers.Dense(256, activation='relu')(x)

In [ ]:
# Metadata Stream (MLP)
meta_input = layers.Input(shape=(train_meta.shape[1],), name="meta_input")
y = layers.Dense(64, activation='relu')(meta_input)
meta_features = layers.Dense(32, activation='relu')(y)

In [ ]:
# Concatenation (The Fusion Point)
combined = layers.Concatenate()([image_features, meta_features])
final_output = layers.Dense(7, activation='softmax')(combined)

model = models.Model(inputs=[image_input, meta_input], outputs=final_output)

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

Training

In [ ]:
model.fit(train_ds, validation_data=val_ds, epochs=10)